# Facets, Guides, and Strips

This page covers the ggh4x-style facet tools exposed through plotnine extension
points: rendered support for inner axes, per-panel scales, manual panel
designs, and axis-guide descriptors, plus strip descriptor classes where direct
drawing depends on plotnine hooks.


In [ ]:
import numpy as np
import pandas as pd
from plotnine_extra import *
from plotnine_extra.data import ToothGrowth, flights, iris, penguins

tooth = ToothGrowth.assign(dose=ToothGrowth["dose"].astype(str))
penguin_data = penguins.dropna(subset=[
    "bill_length_mm",
    "bill_depth_mm",
    "body_mass_g",
    "species",
]).copy()


## Inner axes and per-panel scales

Use `facet_wrap2()` or `facet_grid2()` when a plot needs inner axes or
per-panel position scales. Selector expressions in `scale_x_facet()` and
`scale_y_facet()` are evaluated against facet-layout rows.


In [ ]:
facet_data = pd.DataFrame({
    "x": list(range(6)) + list(range(8, 14)),
    "y": [1, 2, 1.5, 3, 2.7, 3.2, 2, 3, 3.5, 4, 4.5, 5],
    "panel": ["baseline"] * 6 + ["zoom"] * 6,
    "group": ["A", "A", "B", "B", "A", "B"] * 2,
})

(
    ggplot(facet_data, aes("x", "y", color="group"))
    + geom_point(size=2)
    + facet_wrap2("panel", scales="free_x", axes="all")
    + scale_x_continuous(
        guide=guide_axis_manual(
            breaks=[0, 5, 10],
            labels=["low", "mid", "high"],
            label_colour=["#1b9e77", "#7570b3", "#d95f02"],
        )
    )
    + scale_x_facet('panel == "zoom"', limits=(7, 14), breaks=[8, 10, 12])
    + guide_stringlegend(title="Group")
    + theme_clean()
)


## Manual panel designs

`facet_manual()` maps panels to a text design. Repeated labels span cells, and
`#` marks an empty cell.


In [ ]:
manual_data = pd.DataFrame({
    "x": np.tile([1, 2, 3, 4], 3),
    "y": [1, 2, 2.2, 3, 2, 2.5, 3.2, 3.8, 1.2, 1.4, 1.8, 2.6],
    "panel": np.repeat(["wide", "left", "right"], 4),
})

(
    ggplot(manual_data, aes("x", "y"))
    + geom_point(size=2)
    + geom_line()
    + facet_manual("panel", design="""AA
BC""", axes="all")
    + theme_pubclean()
)


## Nested labels and strip descriptors

`guide_axis_nested()` splits axis labels by a delimiter. Nested strip rendering
is available through `facet_nested()` and `facet_nested_wrap()`. The exported
strip descriptor classes store ggh4x-style options; direct strip drawing is
limited by the strip hooks available in the selected plotnine facet.


In [ ]:
nested_data = pd.DataFrame({
    "x": ["A_low", "A_high", "B_low", "B_high"],
    "y": [1.2, 2.4, 1.7, 3.1],
    "family": ["A", "A", "B", "B"],
    "panel": ["one", "one", "two", "two"],
})

(
    ggplot(nested_data, aes("x", "y"))
    + geom_col(fill="#4E79A7")
    + facet_nested_wrap(["family", "panel"], ncol=2, nest_line=True)
    + guide_axis_nested(delim="_")
    + theme_few()
    + labs(x=None, y="Value")
)


In [ ]:
[
    strip_vanilla(clip="off").clip,
    strip_nested(nest_line=True).nest_line,
    strip_themed(text_x=["theme override"]).text_x,
    strip_tag(prefix="(", suffix=")").prefix,
]


## Panel-specific layers

`at_panel()` and `ggsubset()` restrict a layer to panels that match a layout
selector.


In [ ]:
(
    ggplot(facet_data, aes("x", "y"))
    + geom_point(color="#b3b3b3")
    + at_panel(geom_point(color="#d95f02", size=3), 'panel == "zoom"')
    + facet_wrap2("panel", scales="free_x")
    + force_panelsizes(cols=[1.2, 1])
    + theme_clean()
)


## Dendrogram scales

Dendrogram position scales set a categorical order and attach a dendrogram axis
guide. The guide is drawn only when an extended facet renderer can consume it.


In [ ]:
dendro_data = pd.DataFrame({
    "sample": ["C", "A", "B"],
    "value": [3.0, 1.5, 2.4],
})

(
    ggplot(dendro_data, aes("sample", "value"))
    + geom_col(fill="#59A14F")
    + facet_wrap2("sample", scales="free_x")
    + scale_x_dendrogram(hclust={"ivl": ["C", "A", "B"]})
    + theme_tufte()
)
